# Simulating VLTI data

Standalone notebook for the VLTI/PIONIER-like synthetic binary fitting workflow.


In [38]:
# -- uncomment to get interactive plots
# %matplotlib widget
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

import pmoired
print(pmoired.__version__)

1.3.11


## Example B: VLTI/PIONIER-like synthetic data


In [ ]:
# -- VLTI/PIONIER-like synthetic setup
# -- PIONIER operates in H-band; here we use a compact low-resolution sampling
wl = np.linspace(1.53, 1.77, 7)   # wavelength in um (R~40)
lst = np.linspace(16.3, 19.0, 8)  # local sidereal time in hours

# -- AT configurations: choose one or several
pionier_configs = {
    'small':  {'stations': ['A0', 'B2', 'C1', 'D0'], 'doubleDL': False},
    'medium': {'stations': ['A0', 'B5', 'D0', 'G1'], 'doubleDL': False},
    'large':  {'stations': ['A0', 'G1', 'J2', 'K0'], 'doubleDL': False},
    'extended':  {'stations': ['A0', 'B5', 'J2', 'J6'], 'doubleDL': True},
}
selected_configs = ['large']

# -- true binary model (PMOIRED syntax)
# -- A is the primary at (0,0), B is the companion at (B,x, B,y) in mas
model_true = {
    'A,ud': 0.7,
    'A,f': 1.0,
    'B,ud': 0.0,
    'B,f': 0.08,
    'B,x': 3.2,
    'B,y': -2.1,
}

print('Selected configs:', selected_configs)
print('True model:', model_true)


In [ ]:
# -- simulate synthetic OI data from the binary model
# -- target is given as (RA[hours], DEC[deg]) tuple to avoid online name lookup
noise = {
    'V2': 0.01,
    '|V|': 0.01,
    'PHI': 0.5,
    'FLUX': 0.01,
    'T3PHI': 0.1,
    'T3AMP': 0.02,
}

data = []
for name in selected_configs:
    cfg = pionier_configs[name]
    fake = pmoired.oifake.makeFakeVLTI(
        t=cfg['stations'],
        target=(17.5, -30.0),
        lst=lst,
        wl=wl,
        model=model_true,
        noise=noise,
        insname=f'PIONIER_{name}',
        doubleDL=cfg['doubleDL'],
    )
    data.append(fake)
    print(f"{name:>6s}: stations={cfg['stations']} baselines={len(fake['baselines'])} channels={len(fake['WL'])}")


In [ ]:
# -- load synthetic data in PMOIRED OI object
oi = pmoired.OI()
oi.data = data

# -- set observables to fit, similar to PMOIRED examples
oi.setupFit({'obs': ['V2', 'T3PHI']})

# -- show data and true model for reference
oi.showModel(model=model_true, imFov=12, imX=model_true['B,x']/2, imY=model_true['B,y']/2)
oi.show(spectro=False, imFov=12, imX=model_true['B,x']/2, imY=model_true['B,y']/2)


In [ ]:
# -- grid search on companion position (B,x, B,y), similar to PMOIRED examples
maxSep = 8.0
step = 1.0

model_grid = {
    'A,ud': 0.8,
    'A,f': 1.0,
    'B,ud': 0.0,
    'B,f': 0.10,
    'B,x': 0.0,
    'B,y': 0.0,
}

expl = {
    'grid': {
        'B,x': (-maxSep, maxSep, step),
        'B,y': (-maxSep, maxSep, step),
    }
}
prior = [('B,f', '<', 1)]
constrain = [
    ('np.sqrt(B,x**2+B,y**2)', '>', step/2),
    ('np.sqrt(B,x**2+B,y**2)', '<', maxSep),
]

oi.gridFit(expl, model=model_grid, doNotFit=['A,f', 'B,ud'], prior=prior, constrain=constrain)
oi.showGrid(interpolate=True, logV=1, cmap='gist_stern')
print('Grid best model:', {k: oi.bestfit['best'][k] for k in ['A,ud', 'B,f', 'B,x', 'B,y']})

oi.show(spectro=False, imFov=12, imX=model_true['B,x']/2, imY=model_true['B,y']/2)


In [ ]:
# -- fit from an initial guess (intentionally offset from the true parameters)
model_guess = {
    'A,ud': 0.9,
    'A,f': 1.0,
    'B,ud': 0.0,
    'B,f': 0.12,
    'B,x': 1.0,
    'B,y': -4.0,
}

oi.doFit(model_guess, doNotFit=['A,f'])
print('Reduced chi2:', oi.bestfit['chi2'])
print('Best-fit parameters:')
for k in ['A,ud', 'B,f', 'B,x', 'B,y']:
    print(f'  {k:>5s} = {oi.bestfit["best"][k]:.5f}')

oi.show(spectro=False, imFov=12, imX=oi.bestfit['best']['B,x']/2, imY=oi.bestfit['best']['B,y']/2)
